In [21]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver

In [22]:
llm = ChatOllama(model="qwen2.5:0.5b")

In [23]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [24]:
def generate_joke(state: JokeState):

    prompt=f'generate a joke on the topic {state['topic']}'
    response= llm.invoke(prompt).content

    return {'joke': response}

In [25]:
def generate_explanation(state: JokeState):

    prompt=f'write an explanation for joke {state["joke"]} which is on the topic of {state['topic']}'
    response= llm.invoke(prompt).content

    return {'explanation': response}

In [26]:
graph=StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer=InMemorySaver() # to implement persistance

workflow=graph.compile(checkpointer=checkpointer)

In [27]:
config1={"configurable":{"thread_id":1}}
workflow.invoke({'topic':'student'},config=config1)

{'topic': 'student',
 'joke': 'Why did the student break up with the teacher?\n\nBecause he had a new girlfriend!',
 'explanation': 'The joke "Why did the student break up with the teacher? Because he had a new girlfriend!" is a play on words and is often used to make fun of students. In this particular version, the student was playing a game of cat and mouse, and the teacher was his new girlfriend. The punchline is that the student broke up with the teacher because he had a new girlfriend! This play on words is meant to be humorous and could be interpreted in various ways, such as being a play on the idea that "student" means someone who is learning, "teacher" means someone who is teaching, or "student broke up with the teacher" could mean someone who has a relationship with a teacher, among other interpretations.\n\nI hope this explanation helps clarify the joke! If you have any other questions or need further clarification, feel free to ask.'}

In [28]:
workflow.get_state(config=config1)

StateSnapshot(values={'topic': 'student', 'joke': 'Why did the student break up with the teacher?\n\nBecause he had a new girlfriend!', 'explanation': 'The joke "Why did the student break up with the teacher? Because he had a new girlfriend!" is a play on words and is often used to make fun of students. In this particular version, the student was playing a game of cat and mouse, and the teacher was his new girlfriend. The punchline is that the student broke up with the teacher because he had a new girlfriend! This play on words is meant to be humorous and could be interpreted in various ways, such as being a play on the idea that "student" means someone who is learning, "teacher" means someone who is teaching, or "student broke up with the teacher" could mean someone who has a relationship with a teacher, among other interpretations.\n\nI hope this explanation helps clarify the joke! If you have any other questions or need further clarification, feel free to ask.'}, next=(), config={'c

In [29]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'student', 'joke': 'Why did the student break up with the teacher?\n\nBecause he had a new girlfriend!', 'explanation': 'The joke "Why did the student break up with the teacher? Because he had a new girlfriend!" is a play on words and is often used to make fun of students. In this particular version, the student was playing a game of cat and mouse, and the teacher was his new girlfriend. The punchline is that the student broke up with the teacher because he had a new girlfriend! This play on words is meant to be humorous and could be interpreted in various ways, such as being a play on the idea that "student" means someone who is learning, "teacher" means someone who is teaching, or "student broke up with the teacher" could mean someone who has a relationship with a teacher, among other interpretations.\n\nI hope this explanation helps clarify the joke! If you have any other questions or need further clarification, feel free to ask.'}, next=(), config={'

## time travel using persistance

In [34]:
# getting snapshot
workflow.get_state({'configurable': {'thread_id':1,'checkpoint_id': "1f1cded8-7ccc-6723-8000-af5725f6a72c"}})

StateSnapshot(values={'topic': 'student'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1cded8-7ccc-6723-8000-af5725f6a72c'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-10-22T07:52:29.448374+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1cded8-7ccc-6722-bfff-bb4279a55286'}}, tasks=(PregelTask(id='52053af8-0d3f-9708-bc86-c785ad0adf25', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the student break up with the teacher?\n\nBecause he had a new girlfriend!'}),), interrupts=())

In [35]:
# replaying from a specific snapshot
workflow.invoke(None,{'configurable': {'thread_id':1,'checkpoint_id': "1f1cded8-7ccc-6723-8000-af5725f6a72c"}})

{'topic': 'student',
 'joke': "Why don't students go to the gym? Because they have too many classes!",
 'explanation': 'Sure, here\'s an explanation for the joke:\nThe joke is a play on words, which means it\'s combining two distinct ideas into a single phrase or statement. In this case, the phrase "too many classes" is an expression of frustration or disappointment with the amount of classes students have to take each semester.\nThe student is expressing that they have too many classes, and this makes them want to go to the gym. The word "too many" is an idiomatic expression that means "excessive, " "overwhelming, " or "unmanageable. " It\'s a way of expressing that the student has to do too many things to keep up with their studies.\nThe joke is a play on words because the words "classes" and "gym" are used to create a playful contrast. The phrase "too many classes" becomes "too many classes" when it\'s said in a negative way, and the phrase "too many classes" becomes "classes" when 

In [36]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'student', 'joke': "Why don't students go to the gym? Because they have too many classes!", 'explanation': 'Sure, here\'s an explanation for the joke:\nThe joke is a play on words, which means it\'s combining two distinct ideas into a single phrase or statement. In this case, the phrase "too many classes" is an expression of frustration or disappointment with the amount of classes students have to take each semester.\nThe student is expressing that they have too many classes, and this makes them want to go to the gym. The word "too many" is an idiomatic expression that means "excessive, " "overwhelming, " or "unmanageable. " It\'s a way of expressing that the student has to do too many things to keep up with their studies.\nThe joke is a play on words because the words "classes" and "gym" are used to create a playful contrast. The phrase "too many classes" becomes "too many classes" when it\'s said in a negative way, and the phrase "too many classes" bec

#### here in state history the first 4 are of the initial run and the rest are because of reinvoking from a snapshot

In [37]:
# changing the values of the state at perticuler checkpoint
workflow.update_state({'configurable': {'thread_id':1,'checkpoint_id': '1f1cdeca-42da-6610-8000-ece828030548',"checkpoint_ns": ""}},{'topic':'future'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1cdeda-d868-6e27-8000-79400704fd4f'}}

In [38]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'future'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1cdeda-d868-6e27-8000-79400704fd4f'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-10-22T07:53:32.741584+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1cdeca-42da-6610-8000-ece828030548'}}, tasks=(PregelTask(id='53ef4a6f-f0f6-9eea-0afa-cbfc2f569ec0', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'student', 'joke': "Why don't students go to the gym? Because they have too many classes!", 'explanation': 'Sure, here\'s an explanation for the joke:\nThe joke is a play on words, which means it\'s combining two distinct ideas into a single phrase or statement. In this case, the phrase "too many classes" is an expression of frustration or disapp

In [40]:
workflow.invoke(None,{'configurable': {'thread_id':1,'checkpoint_id': "1f1cdeda-d868-6e27-8000-79400704fd4f"}})

{'topic': 'future',
 'joke': 'Why was the tomato red? Because it saw the salad dressing red.',
 'explanation': 'The joke "Why was the tomato red? Because it saw the salad dressing red" is a play on the common idiom "red in tooth and claw," which means something is red and the same as it\'s being shown. In this case, the tomato is being shown as red because it is green, and the salad dressing is being shown as red because it is yellow.\nThe joke plays on the idea that the tomato is a vegetable and the salad dressing is a condiment, so it is not the same vegetable or dressing. The joke is also a play on words, as "red in tooth and claw" is an expression of disgust or disgust with a person or thing. However, in this context, the joke is more about the fact that the tomato and salad dressing are different things, not about the color of the tomato itself.'}

In [41]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'future', 'joke': 'Why was the tomato red? Because it saw the salad dressing red.', 'explanation': 'The joke "Why was the tomato red? Because it saw the salad dressing red" is a play on the common idiom "red in tooth and claw," which means something is red and the same as it\'s being shown. In this case, the tomato is being shown as red because it is green, and the salad dressing is being shown as red because it is yellow.\nThe joke plays on the idea that the tomato is a vegetable and the salad dressing is a condiment, so it is not the same vegetable or dressing. The joke is also a play on words, as "red in tooth and claw" is an expression of disgust or disgust with a person or thing. However, in this context, the joke is more about the fact that the tomato and salad dressing are different things, not about the color of the tomato itself.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1cdedc-3c17-661c-8002